This is me trying to do endogenous regime change now. Hopefully it doesn't take that long? Idk sobbing this all feels exhausting wot am i doing

In [1]:
println("Adding Packages...\n")
using Pkg
Pkg.activate(".")   # so you're using the same packages as me
Pkg.instantiate(; verbose=true)

using JLD2
using Random
using Printf
using LinearAlgebra
using StatsBase: countmap, mean, median
using Dates
using Revise

Adding Packages...



  Activating project at `C:\VAASAVI\Dropbox\Education\OSU\Ongoing_Research\Populism\political-polarization\p`


Now importing the relevant code that I've written: putting this in a different block because that means I can reimport it as needed without much chaos. This takes advantage of the revise module, which checks whether the module has already been loaded and updates if so.

In [2]:
for file in ["src/ModelTypes.jl", "src/Compute.jl", "src/DistrTools.jl", "src/ModelFunctions.jl",
             "src/EGM.jl", "src/Solvers.jl", "src/SteadyState.jl", "src/Predict.jl"]
    modname = Symbol(splitext(basename(file))[1])       # taken from claude, not sure this symbol thing yet
    existed = isdefined(Main, modname)
    includet(file)
    println(existed ? "Updated $file" : "Loaded $file")
end

using .ModelTypes, .Compute, .EGM, .DistrTools, .Solvers, .SteadyState, .Predict, .ModelFunctions

Loaded src/ModelTypes.jl
Loaded src/Compute.jl
Loaded src/DistrTools.jl
Loaded src/ModelFunctions.jl
Loaded src/EGM.jl
Loaded src/Solvers.jl
Loaded src/SteadyState.jl


Loaded src/Predict.jl


Inititalizing the actual environment

In [3]:
# ─── Frequency-invariant parameters ───
const α::Float64 = 0.36
const σ::Float64 = 2
const ϕ::Float64 = 0
const μ_l::Float64 = 0
const μ_z::Float64 = 0

# grid sizes and parameters
const na::Int64 = 100; 
const nl::Int64 = 15;
const nz::Int64 = 2;
const nk::Int64 = 15;

const a_l::Float64 = 0;
const a_h::Float64 = 100;

# policy grids
const np::Int64 = 10; # number of policies
const pol_l::Float64 = 0;
const pol_h::Float64 = .18;

τ_grid = range(pol_l, pol_h, length = np);
η_grid = range(pol_l, pol_h, length = np);
policy_grid = [(η, τ) for η in η_grid, τ in τ_grid];
captax = repeat([0.0], outer = nl);

# ─── Frequency switch: read from environment, default to "annual" ───
const freq = get(ENV, "FREQ", "annual")
display("Frequency set to: $freq")

if freq == "quarterly"
    const β::Float64   = 0.99
    const δ::Float64   = 0.025
    const ρ_l::Float64 = 0.9878      # STY persistence quarterly
    const σ_l::Float64 = 0.087       # STY innovation std, quarterly
    const ρ_z::Float64 = 0.976
    const σ_z::Float64 = 0.007
    
			
elseif freq == "annual"
    const β::Float64   = 0.96
    const δ::Float64   = 0.06
    const ρ_l::Float64 = 0.952     # Storesletten-Telmer-Yaron
    const σ_l::Float64 = 0.17       # = sqrt(0.061), STY persistent innovation std σ_η
    const ρ_z::Float64 = 0.909      # Khan-Thomas 2013
    const σ_z::Float64 = 0.014
    
else
    error("FREQ must be \"annual\" or \"quarterly\", got \"$freq\"")
end

const freq_label = freq;   # use in output filename

"Frequency set to: annual"

Now building the grids

In [4]:
grid_range_l = 2.5     # wide, for near-unit-root labor
grid_range_z = 2.575   # leave z as is

π_l, lgrid = getTauchen(nl, μ_l, σ_l, ρ_l, grid_range_l);
# π_z, zgrid= getTauchen(nz,  μ_z, σ_z, ρ_z, grid_range_z);

# stationary_l = stationary(π_l);
# const lagg::Float64 = dot(stationary_l, lgrid);

zgrid = [0.98, 1.02]          # bad, good
π_z = [0.80  0.20;                 # from bad: 20% chance exit to good
       0.03125  0.96875]          # from good: ~3% chance enter recession

nt = length(LinearIndices(π_z));       # number of states in joint markov chain

stationary_l = stationary(π_l);
const lagg::Float64 = dot(stationary_l, lgrid);
 
# more conservative estimates 
# const kH::Float64 = ((1.0/β - 1.0 + δ) / α)^(1.0/(α-1.0)) * (1.0 + maximum(η_grid)) * 1.5
# const kL::Float64 = max(1.0, ((1.0/β - 1.0 + δ) / α)^(1.0/(α-1.0)) * (1.0 + minimum(η_grid)) * 0.5)

const kL::Float64 = 3; const kH::Float64 = 12; 
Kgrid = collect(range(kL, kH, length = nk));

agrid = logspace(a_l, a_h, na);
amu = collect(range(a_l, a_h, length=na*10));

const NT = 5000; #three thousand periods for sampling
const rnseed = 1234567;

const zt = simz(NT, nz, rnseed, π_z);

const params = ModelParams(α, β, δ, σ, ϕ, agrid, 
		lgrid, zgrid, π_l, π_z, amu, Kgrid);

lgrid

display(sum(stationary(π_l)))

1.0000000000000269

The next thing that needs to be done is like, figuring out what to do to make this endogenous. I'm doing the first part, which is just running it to make sure I can get the forecasts based on z-z transitions, ignoring regime change for now.

In [5]:
Kfore_start = [ 0.151993    0.933069    0.002547; 
    0.1613271876906856 0.932123684433605 3.0517578125e-5; 
    0.160608262813353 0.9324243834645358 3.0517578125e-5; 
    0.151325    0.933213     3.0517578125e-5]

η1 = 0.0; τ = 0.0; captax = repeat([0.0], outer = nl); 
η2 = 0.08; 
policies1 = ProposedPolicies(η1, τ, captax);
policies2 = ProposedPolicies(η2, τ, captax);


r_vals = zeros(nk, nt); w_vals = zeros(nk, nt); λ_vals = zeros(nk, nt); 

for ik = 1:nk, it = 1:nt
	iz = CartesianIndices(π_z)[it][2] # getting today's z'
	r_vals[ik, it] = calcr(α, δ, Kgrid[ik], η1, zgrid[iz]);
	w_vals[ik, it] = calcw(α, Kgrid[ik], η1, zgrid[iz]);
	denom = dot((w_vals[ik, it] .* lgrid).^(1 - τ), stationary(π_l));
	tot_inc = w_vals[ik, it] * dot(lgrid, stationary(π_l));
	λ_vals[ik, it] = tot_inc / denom;
end
prices1 = ImpliedRegimeParams_KS(λ_vals, r_vals, w_vals);

r_vals = zeros(nk, nt); w_vals = zeros(nk, nt); λ_vals = zeros(nk, nt); 

for ik = 1:nk, it = 1:nt
	iz = CartesianIndices(π_z)[it][2] # getting today's z'
	r_vals[ik, it] = calcr(α, δ, Kgrid[ik], η2, zgrid[iz]);
	w_vals[ik, it] = calcw(α, Kgrid[ik], η2, zgrid[iz]);
	denom = dot((w_vals[ik, it] .* lgrid).^(1 - τ), stationary(π_l));
	tot_inc = w_vals[ik, it] * dot(lgrid, stationary(π_l));
	λ_vals[ik, it] = tot_inc / denom;
end
prices2 = ImpliedRegimeParams_KS(λ_vals, r_vals, w_vals);

extrema(prices1.r .- prices2.r)
prices2.r[cld(nk, 2),1]

prices1.r

15×4 Matrix{Float64}:
 0.114651   0.114651   0.12178    0.12178
 0.0942431  0.0942431  0.100539   0.100539
 0.0790061  0.0790061  0.0846798  0.0846798
 0.0671122  0.0671122  0.0723004  0.0723004
 0.0575195  0.0575195  0.0623162  0.0623162
 0.0495868  0.0495868  0.0540597  0.0540597
 0.0428956  0.0428956  0.0470955  0.0470955
 0.0371604  0.0371604  0.0411261  0.0411261
 0.0321789  0.0321789  0.0359413  0.0359413
 0.0278034  0.0278034  0.0313872  0.0313872
 0.0239234  0.0239234  0.0273489  0.0273489
 0.0204545  0.0204545  0.0237383  0.0237383
 0.0173307  0.0173307  0.0204871  0.0204871
 0.0145     0.0145     0.0175408  0.0175408
 0.0119205  0.0119205  0.014856   0.014856

Now we start the actual solution part.

In [6]:
# put this in again here since I'm testing and modifying regularly.

for file in ["src/ModelTypes.jl", "src/Compute.jl", "src/DistrTools.jl", "src/ModelFunctions.jl",
             "src/EGM.jl", "src/Solvers.jl", "src/SteadyState.jl", "src/Predict.jl"]
    modname = Symbol(splitext(basename(file))[1])       # taken from claude, not sure this symbol thing yet
    existed = isdefined(Main, modname)
    includet(file)
    println(existed ? "Updated $file" : "Loaded $file")
end

# init V and friends:
V0 = zeros(nk,nt,nl,na); V  = zeros(nk,nt,nl,na);
G0 = zeros(nk,nt,nl,na); G  = zeros(nk,nt,nl,na);  
C  = zeros(nk,nt,nl,na);

# intializing a starting guess
for ik = 1:nk, it = 1:nt, il = 1:nl, ia = 1:na
	kval = agrid[ia];
	yval = (1 + r_vals[ik, it]*(1-captax[il]))*kval + w_vals[ik, it]*lgrid[il] - r_vals[ik, it]*ϕ;
	ymin = max(1e-10, yval);
	V0[ik, it, il, ia] = log(ymin);
	G0[ik, it, il, ia] = agrid[ia];
end

const dTol = 1e-4;
const vTol = 1e-4;


verbose = true;

Updated src/ModelTypes.jl
Updated src/Compute.jl
Updated src/DistrTools.jl
Updated src/ModelFunctions.jl
Updated src/EGM.jl
Updated src/Solvers.jl
Updated src/SteadyState.jl
Updated src/Predict.jl


In [19]:
# put this in again here since I'm testing and modifying regularly.

for file in ["src/ModelTypes.jl", "src/Compute.jl", "src/DistrTools.jl", "src/ModelFunctions.jl",
             "src/EGM.jl", "src/Solvers.jl", "src/SteadyState.jl", "src/Predict.jl"]
    modname = Symbol(splitext(basename(file))[1])       # taken from claude, not sure this symbol thing yet
    existed = isdefined(Main, modname)
    includet(file)
    println(existed ? "Updated $file" : "Loaded $file")
end

foredist = 10.0
outer_ct = 1
maxout = 25;

Kt = zeros(Float64, NT+1); 
it_t = zeros(Int, NT);
votes_t = zeros(Float64, NT);

Kfore = copy(Kfore_start);

burnin  = 500; λ_damp = 0.3;

# Pkg.add("CodeTracking")
using CodeTracking
using ..DistrTools
methods(summarizeDataByTransition)
println(@code_string summarizeDataByTransition(Kt, it_t, π_z, burn_in)

Updated src/ModelTypes.jl
Updated src/Compute.jl
Updated src/DistrTools.jl
Updated src/ModelFunctions.jl
Updated src/EGM.jl
Updated src/Solvers.jl
Updated src/SteadyState.jl
Updated src/Predict.jl


LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/VAASAVI/Dropbox/Education/OSU/Ongoing_Research/Populism/political-polarization/p/In[19]#27:71\[90mIn[19]:27:71[0;0m]8;;\
methods(summarizeDataByTransition)
println(@code_string summarizeDataByTransition(Kt, it_t, π_z, burn_in)[48;2;120;70;70m[0;0m
[90m#                                                                     └ ── [0;0m[91mExpected `)` or `,`[0;0m

In [20]:
while foredist > dTol && outer_ct ≤ maxout

    # vTol_outer = max(vTol, foredist * 1e-2)
    vTol_outer = vTol;

    # solve HH + simulate under current Kfore
    Kt, it_t, votes_t = genForecastData(V, V0, G, G0, C, Kfore, params, policies1,
                            policies2, prices1, prices2, zt, vTol_outer, verbose = verbose)

    Kfore_new, R2, counts = update_forecast(Kt, it_t, votes_t, nt, burnin)
    foredist = maximum(abs.(Kfore_new[:,1:2] .- Kfore[:,1:2]))

    if verbose
        CI = CartesianIndices(params.π_z)
        println("\nForecast rules:  log K' = a + b·log K")
        println("─"^72)
        @printf("  %-10s %11s %11s %11s %9s %8s\n", "z₋₁→z", "a", "b", "c(Θ)", "R²", "n")
        println("─"^72)
        for z1z2 in 1:nt
            zprev, znow = CI[z1z2][1], CI[z1z2][2]
            flag = (isnan(R2[z1z2]) || R2[z1z2] < 0.99) ? "  ⚠" : ""
            @printf("  %2d→%-6d %11.6f %11.6f %11.6f %9.4f %8d%s\n",
                zprev, znow, Kfore_new[z1z2,1], Kfore_new[z1z2,2],
                Kfore_new[z1z2, 3], R2[z1z2], counts[z1z2], flag)
        end
        println("─"^62)
        @printf("Outer %2i | foredist = %.6f\n", outer_ct, foredist)
        @printf("K range for (%4.2f, %4.2f) vs (%4.2f, %4.2f): %2.4f, %2.4f\n",
            policies1.η, policies1.τ, policies2.η, policies2.τ,
            minimum(Kt[501:end]), maximum(Kt[501:end]))

        @printf("K summary:")
        summarizeDataByTransition(Kt, it_t, params.π_z, burnin)
        @printf("Votes summary:")
        summarizeDataByTransition(votes_t, it_t, params.π_z, burnin)
    end

    # damped update
    Kfore = λ_damp .* Kfore_new .+ (1 - λ_damp) .* Kfore
    outer_ct += 1

end

converged = foredist <= dTol
if converged
    if verbose
        @printf("\nConverged in %i outer iters. foredist = %.6f\n", outer_ct-1, foredist)
    end
else
    # always report failure, regardless of verbose
    @printf("\n Hit maxout=%i without converging. foredist = %.6f\n", maxout, foredist)
    @printf("Maxout at policy (η, τ) = (%4.2f, %4.2f) vs (%4.2f, %4.2f)\n", 
        policies1.η, policies1.τ, policies2.η, policies2.τ)
end

a;ldkfajl

Solving Household Problem...
	Iteration 100: ||V - V0|| = 0.014020, ||G - G0|| = 0.000000, dist = 0.014020
	Iteration 200: ||V - V0|| = 0.000230, ||G - G0|| = 0.000000, dist = 0.000230
	Converged in 222 iters, dist = 0.000097
Solving Household Problem...
	Iteration 100: ||V - V0|| = 0.014165, ||G - G0|| = 0.000000, dist = 0.014165
	Iteration 200: ||V - V0|| = 0.000229, ||G - G0|| = 0.000000, dist = 0.000229
	Converged in 222 iters, dist = 0.000097
	Simulating period 2500 of 5000
	Simulating period 5000 of 5000
vote share: (0.5485967169673027, 0.5493562704391999) mean 0.5487206209665936

Forecast rules:  log K' = a + b·log K
────────────────────────────────────────────────────────────────────────
  z₋₁→z                a           b        c(Θ)        R²        n
────────────────────────────────────────────────────────────────────────
   1→1         0.147049    0.934339    0.002585    1.0000      641
   2→1         0.121564    0.938343    0.032080    1.0000      135
   1→2         0.147

LoadError: InterruptException:

In [9]:
# choose a random starting point for the simulation
ik0 = cld(nk, 2); K0 = Kgrid[ik0];
Kt = zeros(NT+1); Kt[1] = Kgrid[ik0]

# initial distribution: stationary at starting K, lifted to pair-state
G_start = G1[ik0, :, :, :]                        # (nt, nl,na) (base K)

# ... lift to pair-state (nz²) if getDistr is pair-state ...
μ_transit, _ = getDistr(G_start, params.amu, params.agrid, params.π_l, params.π_z,
                     CI, LI, params.ϕ)

#@printf("getDistr: sum(μ_transit) = %.10f  (want 1.0)\n", sum(μ_transit))
#@printf("          min = %.3e  (want ≥ 0, no negatives)\n", minimum(μ_transit))

#@printf("collapse: sum(μ_today) = %.10f  (want = sum(μ_prev))\n", sum(μ_transit))
med_ind = ceil(Int, median(1:length(params.zgrid)))
it_t = zeros(Int, NT);
it_t[1] = LI[med_ind, med_ind]   # initial pair-state index
it_t[2:NT] = [LI[zt[t-1], zt[t]] for t in 2:NT];
μ_transit = μ_transit[it_t[1], :, :]
votes_t = zeros(Float64, NT);

LoadError: UndefVarError: `G1` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [10]:

for t in 1:NT
    if t%2500 == 0 && verbose
        @printf("\tSimulating period %i of %i\n", t, NT)
    end

    # getting today's vote to be used for tomorrow. 
    K = Kt[t]; ix, we = weight(Kgrid, K);
    it = it_t[t] # getting which z transition we're in
    VOTES_t = we.*VOTES[ix, it, :, :] + (1-we) .* VOTES[ix+1, it, :, :];
    _, voteshare = mapVotes(VOTES_t, params, μ_transit)
    votes_t[t] = voteshare;

    # update the distribution for the next period
    G_t = we .* G1[ix, it, :, :] .+ (1-we) .* G1[ix+1, it, :, :];   # (nl, na)
    mass_before = sum(μ_transit)
    μ_transit, Kt[t+1] = transitDistr(G_t, μ_transit, params.amu, params.agrid, params.ϕ, params.π_l)
    mass_after = sum(μ_transit)

    if abs(mass_after - mass_before) > 1e-8
        @printf("LEAK at t=%i: before=%.10f after=%.10f  Δ=%.3e  (K=%.4f)\n",
                t, mass_before, mass_after, mass_after - mass_before, K)
    end
end



LoadError: UndefVarError: `VOTES` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
Kfore_new, R2, counts = update_forecast(Kt, it_t, votes_t, nt, burnin)
foredist = maximum(abs.(Kfore_new[:,1:2] .- Kfore[:,1:2]))

if verbose
    CI = CartesianIndices(params.π_z)
    println("\nForecast rules:  log K' = a + b·log K")
    println("─"^72)
    @printf("  %-10s %11s %11s %11s %9s %8s\n", "z₋₁→z", "a", "b", "c(Θ)", "R²", "n")
    println("─"^72)
    for z1z2 in 1:nt
        zprev, znow = CI[z1z2][1], CI[z1z2][2]
        flag = (isnan(R2[z1z2]) || R2[z1z2] < 0.99) ? "  ⚠" : ""
        @printf("  %2d→%-6d %11.6f %11.6f %11.6f %9.4f %8d%s\n",
            zprev, znow, Kfore_new[z1z2,1], Kfore_new[z1z2,2],
            Kfore_new[z1z2, 3], R2[z1z2], counts[z1z2], flag)
    end
    println("─"^62)
    @printf("Outer %2i | foredist = %.6f\n", outer_ct, foredist)
    @printf("K range for (%4.2f, %4.2f) vs (%4.2f, %4.2f): %2.4f, %2.4f\n",
        policies1.η, policies1.τ, policies2.η, policies2.τ,
        minimum(Kt[501:end]), maximum(Kt[501:end]))

    summarizeKtByTransition(Kt, it_t, params.π_z, burnin)
    summarizeKtByTransition(votes_t, it_t, params.π_z, burnin)
    
    @printf("Vote share extrema: %0.6f, %0.6f\n", extrema(votes_t)...)
end

extrema(votes_t[2:end])
# damped update
# Kfore = λ_damp .* Kfore_new .+ (1 - λ_damp) .* Kfore
# outer_ct += 1

println(length(findall(votes_t .> 0)))        # how many nonzero periods total
println(minimum(findall(votes_t .> 0)))       # earliest period with a vote
println(maximum(findall(votes_t .> 0)))       # latest
println(count(findall(votes_t .> 0) .> 500))  # how many are POST burn-in

println(mean(votes_t))
println(median(votes_t))

println(argmax(votes_t))   # which period has the 0.0097
println(it_t[argmax(votes_t)])  # what pair is it assigned to




Forecast rules:  log K' = a + b·log K
────────────────────────────────────────────────────────────────────────
  z₋₁→z                a           b        c(Θ)        R²        n
────────────────────────────────────────────────────────────────────────
   1→1              NaN         NaN         NaN       NaN      641  ⚠
   2→1              NaN         NaN         NaN       NaN      135  ⚠
   1→2              NaN         NaN         NaN       NaN      135  ⚠
   2→2              NaN         NaN         NaN       NaN     3589  ⚠
──────────────────────────────────────────────────────────────
Outer  1 | foredist = NaN
K range for (0.00, 0.00) vs (0.08, 0.00): 0.0000, 0.0000


LoadError: UndefVarError: `summarizeKtByTransition` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:

println(sum(VOTES == 0))

println(all(V1 .> V2))    # true if V1 strictly dominates everywhere
println(all(V2 .> V1))    # true if V2 strictly dominates everywhere
println(count(V1 .> V2))  # how many states V1 wins
println(count(V2 .> V1))  # how many states V2 wins

println(count(EV1 .> EV2))   # this is what determines VOTES
println(count(EV2 .> EV1))
println(extrema(EV1 .- EV2)) # how big is the gap, and what sign?

println("Extrema of V1")
println(extrema(V1))

println(extrema(prices1.r .- prices2.r))
println(extrema(prices1.w .- prices2.w))
println(extrema(prices1.λ .- prices2.λ))

println(extrema(G1))

# initial distribution: stationary at starting K, lifted to pair-state
G_start = G1[cld(nk, 2), :, :, :]                        # (nt, nl,na) (base K)

# ... lift to pair-state (nz²) if getDistr is pair-state ...
μ_transit, Kagg = getDistr(G_start, params.amu, params.agrid, params.π_l, params.π_z,
                     CI, LI, params.ϕ);
println(Kagg)
println(size(μ_transit))

println(V1 === V2)

LoadError: UndefVarError: `VOTES` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [13]:

# choose a random starting point for the simulation
NT = length(zt);
ik0 = cld(nk, 2); K0 = Kgrid[ik0];
Kt = zeros(NT+1); Kt[1] = Kgrid[ik0]

# initial distribution: stationary at starting K, lifted to pair-state
G_start = G1[ik0, :, :, :]                        # (nt, nl,na) (base K)

# ... lift to pair-state (nz²) if getDistr is pair-state ...
μ_transit, _ = getDistr(G_start, params.amu, params.agrid, params.π_l, params.π_z,
                     CI, LI, params.ϕ)

#@printf("getDistr: sum(μ_prev) = %.10f  (want 1.0)\n", sum(μ_prev))
#@printf("          min = %.3e  (want ≥ 0, no negatives)\n", minimum(μ_prev))

#@printf("collapse: sum(μ_today) = %.10f  (want = sum(μ_prev))\n", sum(μ_today))
med_ind = ceil(Int, median(1:length(params.zgrid)))
it_t = zeros(Int, NT);
it_t[1] = LI[med_ind, med_ind]   # initial pair-state index
it_t[2:NT] = [LI[zt[t-1], zt[t]] for t in 2:NT];
μ_transit = μ_transit[it_t[1], :, :]
votes_t = zeros(Float64, NT);

for t in 1:NT
    if t%2500 == 0 && verbose
        @printf("\tSimulating period %i of %i\n", t, NT)
    end

    # getting today's vote to be used for tomorrow. 
    K = Kt[t]; ix, we = weight(Kgrid, K);
    it = it_t[t] # getting which z transition we're in
    VOTES_t = we.*VOTES[ix, it, :, :] + (1-we) .* VOTES[ix+1, it, :, :];
    _, voteshare = mapVotes(VOTES_t, params, μ_transit)
    votes_t[t] = voteshare;

    # update the distribution for the next period
    G_t = we .* G[ix, it, :, :] .+ (1-we) .* G[ix+1, it, :, :];   # (nl, na)
    mass_before = sum(μ_transit)
    μ_transit, Kt[t+1] = transitDistr(G_t, μ_transit, params.amu, params.agrid, params.ϕ, params.π_l)
    mass_after = sum(μ_transit)

    if abs(mass_after - mass_before) > 1e-8
        @printf("LEAK at t=%i: before=%.10f after=%.10f  Δ=%.3e  (K=%.4f)\n",
                t, mass_before, mass_after, mass_after - mass_before, K)
    end
end

Kfore_new, R2, counts = update_forecast(Kt, it_t, votes_t, nt, burnin)
foredist = maximum(abs.(Kfore_new[:,1:2] .- Kfore[:,1:2]))

if verbose
    CI = CartesianIndices(params.π_z)
    println("\nForecast rules:  log K' = a + b·log K")
    println("─"^72)
    @printf("  %-10s %11s %11s %11s %9s %8s\n", "z₋₁→z", "a", "b", "c(Θ)", "R²", "n")
    println("─"^72)
    for z1z2 in 1:nt
        zprev, znow = CI[z1z2][1], CI[z1z2][2]
        flag = (isnan(R2[z1z2]) || R2[z1z2] < 0.99) ? "  ⚠" : ""
        @printf("  %2d→%-6d %11.6f %11.6f %11.6f %9.4f %8d%s\n",
            zprev, znow, Kfore_new[z1z2,1], Kfore_new[z1z2,2],
            Kfore_new[z1z2, 3], R2[z1z2], counts[z1z2], flag)
    end
    println("─"^62)
    @printf("Outer %2i | foredist = %.6f\n", outer_ct, foredist)
    @printf("K range for (%4.2f, %4.2f) vs (%4.2f, %4.2f): %2.4f, %2.4f\n",
        policies1.η, policies1.τ, policies2.η, policies2.τ,
        minimum(Kt[501:end]), maximum(Kt[501:end]))

    summarizeKtByTransition(Kt, it_t, params.π_z, burnin)
end

# damped update
Kfore = λ_damp .* Kfore_new .+ (1 - λ_damp) .* Kfore
outer_ct += 1

LoadError: invalid assignment to constant Main.NT. This redefinition may be permitted using the `const` keyword.